# Part 4 - Generative AI for reasoning about missed appointments
We send the call-centre follow-up notes (first 10 only) to **Google Gemini**, force the answer into a
**Pydantic schema** (`root_cause`, `confidence`, `summary`, `recommended_action`), validate it,
and combine it with the **risk score from Part 3**.

**Before you run:** you need a free API key from https://aistudio.google.com (Get API key).
**Never paste your key into a cell that you save to GitHub.** The key cell below asks for it safely.

## Task 4.1 - Gemini API integration

In [ ]:
# Install the current Google Gen AI SDK (package name: google-genai) and pydantic.
# (The older package "google-generativeai" is the legacy library; google-genai replaces it.)
%pip install -q -U google-genai pydantic

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/YOUR-USERNAME/YOUR-REPO-NAME.git"   # only used in Colab if files are missing
if not os.path.exists("all_risk_scores.csv"):
    subprocess.run(["git", "clone", REPO_URL, "project"], check=True)
    os.chdir("project")
print("Working folder:", os.getcwd())

In [ ]:
# ---- API key: kept in memory only, never written to a file ----
# Colab: click the key icon in the left sidebar > "Add new secret" > Name: GEMINI_API_KEY > paste key > enable notebook access.
# Anywhere else: you will be asked to paste the key.
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    import getpass
    API_KEY = getpass.getpass("Paste your Gemini API key: ")

from google import genai
from google.genai import types

client = genai.Client(api_key=API_KEY)

In [ ]:
# Which models can YOUR key use? (Model names change over time, so we look them up instead of guessing.)
available = [m.name for m in client.models.list() if "flash" in m.name.lower()]
print("Flash models available to your key:")
for name in available:
    print("  ", name)

In [ ]:
# Pick a model from the list above. Names are written WITHOUT the "models/" prefix.
# If the call later fails with "not found" / "404", change this to a name printed above.
MODEL_NAME = "gemini-2.5-flash"

# Thinking level: the brief says "as per session guidelines" - set it to whatever your instructor said.
# "low" keeps answers fast and cheap. Older models do not support this setting; the code below
# automatically retries without it if the API complains.
THINKING_LEVEL = "low"

In [ ]:
from typing import Literal
from pydantic import BaseModel, ValidationError, field_validator

RootCause = Literal[
    "reminder_not_received",   # never got / forgot because no reminder
    "distance_transport",      # too far, no transport
    "cost_financial",          # fee too high, money problem
    "illness",                 # patient unwell
    "work_conflict",           # job, shift, meeting
    "contact_unreachable",     # phone off, wrong number
    "other",                   # a reason not covered above
    "no_information",          # note empty or says nothing about why
]


class NoShowAnalysis(BaseModel):
    root_cause: RootCause
    confidence: float          # 0.0 (guess) to 1.0 (note states it clearly)
    summary: str
    recommended_action: str

    @field_validator("confidence")
    @classmethod
    def confidence_in_range(cls, v):
        if not 0.0 <= v <= 1.0:
            raise ValueError("confidence must be between 0 and 1")
        return v

    @field_validator("summary", "recommended_action")
    @classmethod
    def not_blank(cls, v):
        if not v or not v.strip():
            raise ValueError("text field must not be blank")
        return v.strip()

## Task 4.2 - `analyze_note()`

In [ ]:
import time

SYSTEM_INSTRUCTION = "You are an AI assistant analyzing patient appointment data"

PROMPT_TEMPLATE = """A patient missed a clinic appointment. A call-centre agent wrote this follow-up note
(it may be in English or Hinglish):

\"\"\"{note}\"\"\"

Return JSON with:
- root_cause: the single best category for WHY the appointment was missed
- confidence: a number from 0 to 1 - how clearly the note supports that category
- summary: one short English sentence describing what happened
- recommended_action: one practical step the clinic should take next
Use only what the note says. If the note does not say why, use "no_information" with low confidence."""

USE_THINKING = True


def make_config():
    cfg = dict(
        system_instruction=SYSTEM_INSTRUCTION,
        temperature=0,
        response_mime_type="application/json",
        response_schema=NoShowAnalysis,
        max_output_tokens=1500,          # brief asks for 1000+
    )
    if USE_THINKING:
        cfg["thinking_config"] = types.ThinkingConfig(thinking_level=THINKING_LEVEL)
    return types.GenerateContentConfig(**cfg)


def analyze_note(note_text, retries=3):
    """Analyse one note. Returns (NoShowAnalysis or None, status, error_message)."""
    global USE_THINKING

    # Edge case: nothing to analyse -> do not waste an API call
    if not isinstance(note_text, str) or not note_text.strip():
        empty = NoShowAnalysis(
            root_cause="no_information", confidence=0.0,
            summary="No follow-up note text was recorded.",
            recommended_action="Call the patient again and record the reason for the missed visit.")
        return empty, "skipped_empty_note", ""

    last_error = ""
    for attempt in range(1, retries + 1):
        try:
            resp = client.models.generate_content(
                model=MODEL_NAME,
                contents=PROMPT_TEMPLATE.format(note=note_text.strip()[:2000]),
                config=make_config())
            return NoShowAnalysis.model_validate_json(resp.text), "ok", ""
        except ValidationError as err:            # the model answered, but not in our schema
            last_error = f"validation error: {err.error_count()} problem(s)"
        except Exception as err:                  # network, quota, wrong model name ...
            last_error = f"{type(err).__name__}: {err}"
            if USE_THINKING and "thinking" in str(err).lower():
                print("  (this model does not accept the thinking setting - retrying without it)")
                USE_THINKING = False
                continue
        print(f"  attempt {attempt}/{retries} failed - {last_error[:150]}")
        time.sleep(10 * attempt)                  # free tier has a per-minute limit; wait and retry
    return None, "failed", last_error

In [ ]:
# Quick live test on ONE note before running all ten
test_result, status, err = analyze_note("Patient said she never got any reminder message and forgot the visit.")
print(status, err)
print(test_result)

In [ ]:
import pandas as pd

notes = pd.read_csv("data/follow_up_notes.csv").head(10)     # brief: first 10 notes only
print("Notes to process:", len(notes))

records = []
for _, row in notes.iterrows():
    result, status, err = analyze_note(row["note_text"])
    print(row["note_id"], row["appointment_id"], "->", status,
          result.root_cause if result else "-")
    records.append({
        "note_id": row["note_id"],
        "appointment_id": row["appointment_id"],
        "note_text": row["note_text"],
        "root_cause": result.root_cause if result else None,
        "confidence": result.confidence if result else None,
        "summary": result.summary if result else None,
        "recommended_action": result.recommended_action if result else None,
        "analysis_status": status,
        "error": err,
    })
    time.sleep(7)       # stay under the free-tier requests-per-minute limit

analysis_df = pd.DataFrame(records)
analysis_df[["note_id", "root_cause", "confidence", "summary", "analysis_status"]]

## Task 4.3 - Validation and error handling
Part A tests the Pydantic validator directly with deliberately broken "model answers" (no API call needed).
Part B tests `analyze_note()` with bad inputs.

In [ ]:
bad_answers = {
    "valid answer": '{"root_cause":"illness","confidence":0.9,"summary":"Patient was sick.","recommended_action":"Rebook."}',
    "confidence above 1": '{"root_cause":"illness","confidence":1.7,"summary":"Sick.","recommended_action":"Rebook."}',
    "unknown root_cause": '{"root_cause":"aliens","confidence":0.5,"summary":"x","recommended_action":"y"}',
    "missing field": '{"root_cause":"illness","confidence":0.5,"summary":"Sick."}',
    "blank summary": '{"root_cause":"illness","confidence":0.5,"summary":"   ","recommended_action":"Rebook."}',
    "not JSON at all": "Sorry, I cannot help with that.",
}

for label, raw in bad_answers.items():
    try:
        NoShowAnalysis.model_validate_json(raw)
        print(f"{label:20s} -> ACCEPTED")
    except ValidationError as err:
        print(f"{label:20s} -> REJECTED ({err.error_count()} problem(s))")

In [ ]:
bad_inputs = {
    "None": None,
    "empty string": "",
    "only spaces": "     ",
    "a number": 12345,
    "gibberish": "asdf qwer zxcv 1234 !!!! ????",
}
for label, value in bad_inputs.items():
    result, status, err = analyze_note(value)
    print(f"{label:14s} -> status={status:18s} root_cause={result.root_cause if result else None}"
          f" confidence={result.confidence if result else None}")

## Task 4.4 - Final recommendation (risk score + AI root cause + confidence)
Part 3 scored every booking. We attach each note's appointment risk score, then apply simple, transparent rules:
* Low AI confidence (< 0.6) or a failed analysis -> **Manual review**
* Otherwise the priority follows the risk band (High / Medium / Low)

The playbook text below is a starting point written for this project - edit it to match real clinic policy.

In [ ]:
risk = pd.read_csv("all_risk_scores.csv")[["appointment_id", "clinic_id", "risk_score", "risk_band", "split"]]
risk = risk.rename(columns={"split": "score_split"})

final = analysis_df.merge(risk, on="appointment_id", how="left")

PLAYBOOK = {
    "reminder_not_received": "Send SMS/WhatsApp reminders 48h and 2h before the visit and verify the phone number.",
    "distance_transport": "Offer a nearer clinic, a teleconsult, or a more convenient slot.",
    "cost_financial": "Explain fee options or follow-up discounts if policy allows; do not drop the patient.",
    "illness": "Offer a quick reschedule and a check-in call after recovery.",
    "work_conflict": "Offer early-morning, evening or weekend slots.",
    "contact_unreachable": "Correct the contact details and try an alternate number or channel.",
    "other": "Agent to review the note and decide the follow-up.",
    "no_information": "Call the patient again and record the reason for the missed visit.",
}


def final_recommendation(r):
    if r["analysis_status"] == "failed" or pd.isna(r["confidence"]) or r["confidence"] < 0.6:
        priority = "Manual review"
    else:
        priority = {"High": "High priority", "Medium": "Medium priority", "Low": "Standard"}.get(r["risk_band"], "Manual review")
    base = PLAYBOOK.get(r["root_cause"], PLAYBOOK["other"])
    score = f"{r['risk_score']:.2f} ({r['risk_band']})" if pd.notna(r["risk_score"]) else "n/a"
    conf = f"{r['confidence']:.2f}" if pd.notna(r["confidence"]) else "n/a"
    return f"[{priority}] {base} | Risk score {score}, AI confidence {conf}."


final["final_recommendation"] = final.apply(final_recommendation, axis=1)

out_cols = ["note_id", "appointment_id", "clinic_id", "risk_score", "risk_band", "score_split",
            "note_text", "root_cause", "confidence", "summary", "recommended_action",
            "analysis_status", "final_recommendation"]
final[out_cols].to_csv("10_analyzed_notes.csv", index=False)
print("Saved 10_analyzed_notes.csv ->", final[out_cols].shape)
final[["note_id", "root_cause", "confidence", "risk_score", "risk_band", "final_recommendation"]]